# Ultramarathon Performance Visualizations — Part A
**Group 17 — Luiz Samelo, Maximilian Staudacher, Rayudu Muralikrishna**  
*Information Visualization (VU 2.0), TU Wien, 2026*

This notebook covers:
- **Viz 1**: Choropleth map — avg pace by country with year slider
- **Viz 2**: Top 10 national dominance line chart

## 0. Imports & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import country_converter as coco
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../data/TWO_CENTURIES_OF_UM_RACES.csv')
print('Shape:', df.shape)
df.head()

## 1. Data Cleaning & Preprocessing

In [ ]:
# Rename columns for easier access
df.columns = [
    'year', 'race_name', 'race_length', 'num_finishers',
    'athlete_performance', 'athlete_club', 'athlete_country',
    'athlete_year_of_birth', 'athlete_gender', 'athlete_age_category',
    'athlete_avg_speed', 'athlete_id'
]

# Convert types
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['athlete_avg_speed'] = pd.to_numeric(df['athlete_avg_speed'], errors='coerce')

# Filter: keep only 50km races, years 1970–2022, valid speed & country
df_clean = df[
    (df['race_length'] == '50km') &
    (df['year'] >= 1970) &
    (df['year'] <= 2022) &
    (df['athlete_avg_speed'] > 0) &
    (df['athlete_country'].notna()) &
    (df['athlete_country'].str.len() == 3)  # ISO3 codes only
].copy()

print('Cleaned shape:', df_clean.shape)
print('Year range:', df_clean['year'].min(), '–', df_clean['year'].max())
print('Countries:', df_clean['athlete_country'].nunique())

---
## Viz 1 — Choropleth Map: Avg Speed by Country with Year Slider
**Research angle**: How has average 50km race speed varied geographically over the decades?

In [ ]:
# Aggregate: avg speed per country per year (min 10 finishers for reliability)
choropleth_data = (
    df_clean
    .groupby(['year', 'athlete_country'])
    .agg(
        avg_speed=('athlete_avg_speed', 'mean'),
        finishers=('athlete_avg_speed', 'count')
    )
    .reset_index()
)

# Keep only country-years with at least 10 finishers
choropleth_data = choropleth_data[choropleth_data['finishers'] >= 10].copy()
choropleth_data['avg_speed'] = choropleth_data['avg_speed'].round(2)

# Convert ISO3 to country name for hover labels
cc = coco.CountryConverter()
choropleth_data['country_name'] = cc.pandas_convert(
    series=choropleth_data['athlete_country'],
    to='name_short',
    not_found=None
)

print(choropleth_data.shape)
choropleth_data.head()

In [ ]:
# Build choropleth with animation frame = year
fig1 = px.choropleth(
    choropleth_data,
    locations='athlete_country',
    locationmode='ISO-3',
    color='avg_speed',
    animation_frame='year',
    color_continuous_scale='RdYlGn',
    range_color=[6, 14],
    hover_name='country_name',
    hover_data={
        'avg_speed': ':.2f',
        'finishers': True,
        'athlete_country': False,
        'year': False
    },
    labels={
        'avg_speed': 'Avg Speed (km/h)',
        'finishers': 'Finishers'
    },
    title='Average 50km Ultra-Marathon Speed by Country (1970–2022)'
)

fig1.update_layout(
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='natural earth'
    ),
    coloraxis_colorbar=dict(
        title='Avg Speed<br>(km/h)',
        tickvals=[6, 8, 10, 12, 14],
    ),
    title_font_size=18,
    height=550,
    margin=dict(l=0, r=0, t=60, b=0),
    updatemenus=[dict(
        type='buttons',
        showactive=False,
        y=0,
        x=0.5,
        xanchor='center',
        buttons=[dict(
            label='▶ Play',
            method='animate',
            args=[None, dict(frame=dict(duration=400, redraw=True), fromcurrent=True)]
        )]
    )]
)

fig1.show()
fig1.write_html('../Viz/viz1_choropleth_speed_by_country.html')
print('Saved to Viz/viz1_choropleth_speed_by_country.html')

---
## Viz 2 — Top 10 National Dominance Line Chart
**Research angle**: Which countries have dominated ultra-marathon finishing over the decades, and how has that shifted?

In [ ]:
# Use all distances for dominance (broader picture)
df_all = df[
    (df['year'] >= 1970) &
    (df['year'] <= 2022) &
    (df['athlete_country'].notna()) &
    (df['athlete_country'].str.len() == 3)
].copy()

df_all['year'] = pd.to_numeric(df_all['year'], errors='coerce')

# Count finishers per country per year
dominance = (
    df_all
    .groupby(['year', 'athlete_country'])
    .size()
    .reset_index(name='finishers')
)

# Find overall top 10 countries by total finishers across all years
top10_countries = (
    dominance
    .groupby('athlete_country')['finishers']
    .sum()
    .nlargest(10)
    .index.tolist()
)

print('Top 10 countries:', top10_countries)

# Filter to top 10 only
dominance_top10 = dominance[dominance['athlete_country'].isin(top10_countries)].copy()

# Convert ISO3 to country names
dominance_top10['country_name'] = cc.pandas_convert(
    series=dominance_top10['athlete_country'],
    to='name_short',
    not_found=dominance_top10['athlete_country']
)

# Smooth with 3-year rolling average
dominance_top10 = dominance_top10.sort_values(['athlete_country', 'year'])
dominance_top10['finishers_smooth'] = (
    dominance_top10
    .groupby('athlete_country')['finishers']
    .transform(lambda x: x.rolling(3, min_periods=1).mean())
)

dominance_top10.head(10)

In [ ]:
# Build line chart
fig2 = px.line(
    dominance_top10,
    x='year',
    y='finishers_smooth',
    color='country_name',
    hover_data={
        'finishers': True,
        'finishers_smooth': ':.0f',
        'athlete_country': False
    },
    labels={
        'year': 'Year',
        'finishers_smooth': 'Finishers (3-yr avg)',
        'country_name': 'Country',
        'finishers': 'Raw finishers'
    },
    title='Top 10 Countries by Ultra-Marathon Finishers Over Time (1970–2022)',
    color_discrete_sequence=px.colors.qualitative.Bold
)

# Add decade annotations
for decade in [1980, 1990, 2000, 2010, 2020]:
    fig2.add_vline(
        x=decade,
        line_dash='dot',
        line_color='gray',
        opacity=0.5
    )
    fig2.add_annotation(
        x=decade,
        y=1,
        yref='paper',
        text=str(decade),
        showarrow=False,
        font=dict(size=10, color='gray'),
        yanchor='top'
    )

fig2.update_layout(
    hovermode='x unified',
    legend=dict(
        title='Country',
        orientation='v',
        x=1.01,
        y=1
    ),
    xaxis=dict(title='Year', dtick=5),
    yaxis=dict(title='Number of Finishers (3-yr rolling avg)'),
    title_font_size=18,
    height=550,
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig2.update_xaxes(showgrid=True, gridcolor='#f0f0f0')
fig2.update_yaxes(showgrid=True, gridcolor='#f0f0f0')

fig2.show()
fig2.write_html('../Viz/viz2_national_dominance.html')
print('Saved to Viz/viz2_national_dominance.html')

---
## Summary

| Viz | File saved |
|-----|------------|
| 1 — Choropleth map | `Viz/viz1_choropleth_speed_by_country.html` |
| 2 — National dominance line chart | `Viz/viz2_national_dominance.html` |